In [13]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [14]:
df=pd.read_csv('data_for_model_training.csv')



In [15]:
df.head(1)

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4,0,0.0,0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0


In [ ]:
const payload = {
  male: 1,
  age: 45,
  cigsPerDay: 0,
  BPMeds: 0,
  prevalentHyp: 0,
  diabetes: 0,
  totChol: 210,
  sysBP: 128,
  BMI: 24.6,
  glucose: 92,
};


In [16]:
selected_features = [
    'male',
    'age',
    'cigsPerDay',
    'BPMeds',
    'prevalentHyp',
    'diabetes',
    'totChol',
    'sysBP',
    'BMI',
    'glucose',
    'TenYearCHD'
]

df_model = df[selected_features]
df_model.head()

,male,age,cigsPerDay,BPMeds,prevalentHyp,diabetes,totChol,sysBP,BMI,glucose,TenYearCHD
0,1,39,0.0,0,0,0,195.0,106.0,26.97,77.0,0
1,0,46,0.0,0,0,0,250.0,121.0,28.73,76.0,0
2,1,48,20.0,0,0,0,245.0,127.5,25.34,70.0,0
3,0,61,30.0,0,1,0,225.0,150.0,28.58,103.0,1
4,0,46,23.0,0,0,0,285.0,130.0,23.10,85.0,0


In [17]:
df_model.rename(columns={"male": "gender"}, inplace=True)


C:\Users\heamr\AppData\Local\Temp\ipykernel_17100\3672268357.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model.rename(columns={"male": "gender"}, inplace=True)


In [18]:
len(df_model["BPMeds"].unique()) == 2

True

In [19]:
unique=[]

for col in df_model:
    if len(df_model[col].unique()) == 2:
        unique.append(col)
unique

['gender', 'BPMeds', 'prevalentHyp', 'diabetes', 'TenYearCHD']

In [20]:
import pickle as pkl
X=df_model.drop('TenYearCHD', axis=1)
y=df_model['TenYearCHD']
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
xtrain_scaled = scaler.fit_transform(xtrain)
xtest_scaled = scaler.transform(xtest)

with open('scaler.pkl', 'wb') as f:
    pkl.dump(scaler, f)

In [21]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(xtrain_scaled, ytrain)
y_pred_proba = lr.predict_proba(xtest_scaled)[:, 1]

model_bundle = {
    'model':     lr,
    'scaler':    scaler,       # include if you used StandardScaler
    'threshold': 0.3
}

with open('model_bundle.pkl', 'wb') as f:
    pkl.dump(model_bundle, f)

In [23]:
# Load
import pickle


with open('model_bundle.pkl', 'rb') as f:
    bundle = pickle.load(f)

lr        = bundle['model']
scaler    = bundle['scaler']
THRESHOLD = bundle['threshold']   # 0.2

# Predict with your custom threshold
def predict(X):
    X_scaled = scaler.transform(X)
    proba    = lr.predict_proba(X_scaled)[:, 1]
    return (proba >= THRESHOLD).astype(int)

# Usage
y_pred = predict(xtest)

In [ ]:
y_pred = (y_pred_proba >= 0.3).astype(int)
    
resul={
        'Threshold': 0.3,
        'Recall': recall_score(ytest, y_pred),
        'Precision': precision_score(ytest, y_pred, zero_division=0),
        'F1 Score': f1_score(ytest, y_pred),
        'Accuracy': accuracy_score(ytest, y_pred),
        'ROC AUC': roc_auc_score(ytest, y_pred_proba)  # ✅ fixed
    }

print(resul)

{'Threshold': 0.3, 'Recall': 0.9349593495934959, 'Precision': 0.19327731092436976, 'F1 Score': 0.3203342618384401, 'Accuracy': 0.42452830188679247, 'ROC AUC': 0.7146285393888422}


In [ ]:
# Assuming xtrain, ytrain, xtest, ytest are already defined
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(xtrain_scaled, ytrain)
y_pred_proba = lr.predict_proba(xtest_scaled)[:, 1]

thresholds = [0.1 * i for i in range(1, 10)]
results = []

for t in thresholds:
    y_pred = (y_pred_proba >= t).astype(int)
    
    results.append({
        'Threshold': t,
        'Recall': recall_score(ytest, y_pred),
        'Precision': precision_score(ytest, y_pred, zero_division=0),
        'F1 Score': f1_score(ytest, y_pred),
        'Accuracy': accuracy_score(ytest, y_pred),
        'ROC AUC': roc_auc_score(ytest, y_pred_proba)  # ✅ fixed
    })

results_df = pd.DataFrame(results)
display(results_df)


,Threshold,Recall,Precision,F1 Score,Accuracy,ROC AUC
0,0.1,1.000000,0.145218,0.253608,0.146226,0.714629
1,0.2,0.967480,0.161028,0.276102,0.264151,0.714629
2,0.3,0.934959,0.193277,0.320334,0.424528,0.714629
3,0.4,0.780488,0.214286,0.336252,0.553066,0.714629
4,0.5,0.626016,0.248387,0.355658,0.670991,0.714629
5,0.6,0.414634,0.267016,0.324841,0.750000,0.714629
6,0.7,0.227642,0.304348,0.260465,0.812500,0.714629
7,0.8,0.081301,0.400000,0.135135,0.849057,0.714629
8,0.9,0.008130,1.000000,0.016129,0.856132,0.714629
